In [1]:
from pathlib import Path
import xarray as xr
import sys

sys.path.append(str(Path("../src").resolve()))
from radar_lwc.retrieval.gas_attenuation import specific_gas_attenuation, alpha_field

In [2]:
Processed_Dataset = "20261002_pallas_gaussian"

In [3]:
in_path = Path(f"../data/processed/{Processed_Dataset}/aligned_LWP2.nc")
ds = xr.open_dataset(in_path)

In [4]:
alpha_W = alpha_field(
    P_field=ds["P_cloud"],
    T_field=ds["T_cloud"],
    AH_field=ds["AH_cloud"],
    freq_ghz=94.0,
)

alpha_Ka = alpha_field(
    P_field=ds["P_cloud"],
    T_field=ds["T_cloud"],
    AH_field=ds["AH_cloud"],
    freq_ghz=35.0,
)

In [5]:
print(alpha_W)
print(alpha_Ka)

print(f"W-band:  {float(alpha_W.min()):.3f} - {float(alpha_W.max()):.3f} dB/km")
print(f"Ka-band: {float(alpha_Ka.min()):.3f} - {float(alpha_Ka.max()):.3f} dB/km")

<xarray.DataArray 'alpha_94GHz' (Time: 34688, range: 383)> Size: 106MB
array([[0.40586831, 0.40012596, 0.39310754, ...,        nan,        nan,
               nan],
       [0.40588518, 0.40014188, 0.39312269, ...,        nan,        nan,
               nan],
       [0.40590197, 0.40015773, 0.39313778, ...,        nan,        nan,
               nan],
       ...,
       [0.50349066, 0.4966876 , 0.49008573, ...,        nan,        nan,
               nan],
       [0.50349066, 0.4966876 , 0.49008573, ...,        nan,        nan,
               nan],
       [0.50349066, 0.4966876 , 0.49008573, ...,        nan,        nan,
               nan]], shape=(34688, 383))
Coordinates:
  * Time     (Time) datetime64[ns] 278kB 2026-10-02T00:46:20.572915077 ... 20...
  * range    (range) float64 3kB 216.0 252.0 288.0 ... 1.094e+04 1.097e+04
Attributes:
    units:      dB km-1
    long_name:  Gaseous attenuation at 94.0 GHz
    model:      ITU-R P.676
<xarray.DataArray 'alpha_35GHz' (Time: 34688, range

In [6]:
ds_alpha = xr.Dataset(
    {
        "alpha_W": alpha_W,
        "alpha_Ka": alpha_Ka,
    },
    attrs={
        "description": "Gas attenuation computed with ITU-R P.676-13",
        "created_by": "02_05_gas_attenuation.ipynb",
    },
)

ds_alpha.to_netcdf(f"../data/processed/{Processed_Dataset}/gas_attenuation.nc")